In [0]:
# Celda 1
dbutils.widgets.removeAll()

In [0]:
# Celda 2
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F
from pyspark.sql import Window
import re

In [0]:
# Celda 3
dbutils.widgets.text("catalogo", "catalog_au")
dbutils.widgets.text("esquema_source", "bronze")
dbutils.widgets.text("esquema_sink", "silver")

In [0]:
# Celda 4
catalogo = dbutils.widgets.get("catalogo")
esquema_source = dbutils.widgets.get("esquema_source")
esquema_sink = dbutils.widgets.get("esquema_sink")

In [0]:
# Celda 5 - UDFs de limpieza
def normalize_title(s):
    if s is None:
        return None
    s = s.lower().strip()
    s = re.sub(r'\(.*?\)', '', s)
    s = s.split(':')[0]
    s = s.split('/')[0]
    s = re.sub(r'[^a-z0-9 ]', '', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s

def normalize_author(s):
    if s is None:
        return None
    s = s.lower().strip()
    s = re.sub(r'[^a-z0-9 ]', '', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s

normalize_title_udf = F.udf(normalize_title, StringType())
normalize_author_udf = F.udf(normalize_author, StringType())

In [0]:
# Celda 6
df_catalog = spark.table(f"{catalogo}.{esquema_source}.goodreads_books")

In [0]:
# Celda 7
df_catalog = df_catalog.dropna(how="all") \
                        .filter((col("title").isNotNull()) & (col("author").isNotNull()))

In [0]:
# Celda 8
df_catalog = df_catalog.withColumn(
    "primary_genre", F.regexp_extract(col("genres"), r"'([^']+)'", 1)
).withColumn(
    "genres_clean", F.regexp_replace(col("genres"), r"[\[\]']", "")
)

# Ajuste dentro de la Celda 8 (o donde prefieras, antes del select final)
df_catalog = df_catalog.withColumn(
    "isbn",
    F.when(col("isbn").rlike(r'^\d\.\d+E\+\d+$'),
           col("isbn").cast("decimal(20,0)").cast("string"))
     .otherwise(col("isbn"))
)

In [0]:
# Celda 9 (corregida con try_to_date para tolerar errores de parseo)
df_catalog = df_catalog.withColumn(
    "publishDate_clean",
    F.regexp_replace(col("publishDate"), r'(\d+)(st|nd|rd|th)', r'$1')
)

df_catalog = df_catalog.withColumn("mm_2d", F.regexp_extract(col("publishDate"), r'^(\d{1,2})/(\d{1,2})/(\d{2})$', 1)) \
                        .withColumn("dd_2d", F.regexp_extract(col("publishDate"), r'^(\d{1,2})/(\d{1,2})/(\d{2})$', 2)) \
                        .withColumn("yy_2d", F.regexp_extract(col("publishDate"), r'^(\d{1,2})/(\d{1,2})/(\d{2})$', 3))

df_catalog = df_catalog.withColumn(
    "publishDate_2digit_fixed",
    F.when(
        col("yy_2d") != "",
        F.concat(
            F.when(col("yy_2d").cast("int") <= 26, F.lit("20")).otherwise(F.lit("19")),
            col("yy_2d"), F.lit("-"), F.lpad(col("mm_2d"), 2, "0"), F.lit("-"), F.lpad(col("dd_2d"), 2, "0")
        )
    )
)

df_catalog = df_catalog.withColumn(
    "publish_date",
    F.coalesce(
        F.expr("try_to_date(publishDate_2digit_fixed, 'yyyy-MM-dd')"),
        F.expr("try_to_date(publishDate_clean, 'MMMM d yyyy')"),
        F.expr("try_to_date(publishDate_clean, 'MMM d yyyy')"),
        F.expr("try_to_date(publishDate, 'MM-dd-yyyy')"),
        F.expr("try_to_date(publishDate, 'yyyy')")
    )
).drop("publishDate_clean", "mm_2d", "dd_2d", "yy_2d", "publishDate_2digit_fixed")

In [0]:
# Celda 10
df_catalog = df_catalog.withColumn("title_norm", normalize_title_udf(col("title"))) \
                        .withColumn("author_norm", normalize_author_udf(col("author"))) 

In [0]:
# Celda 11
df_catalog_final = df_catalog.select(
    col("title"),
    col("series"),
    col("author"),
    col("rating"),
    col("language"),
    col("isbn"),
    col("primary_genre"),
    col("genres_clean"),
    col("pages"),
    col("publisher"),
    col("publish_date"),
    col("numRatings").alias("num_ratings"),
    col("likedPercent").alias("liked_percent"),
    col("price")
).withColumn("ingestion_date", current_timestamp())

In [0]:
# Celda 12 - Write a Silver
df_catalog_final.write.mode("overwrite").insertInto(f"{catalogo}.{esquema_sink}.books_catalog_transformed")

In [0]:
# Celda 13
df_bestsellers = spark.table(f"{catalogo}.{esquema_source}.amazon_bestsellers")

In [0]:
# Celda 14
df_bestsellers = df_bestsellers.dropna(how="all") \
                                .filter((col("name").isNotNull()) & (col("author").isNotNull()))

In [0]:
# Celda 15
df_bestsellers_final = df_bestsellers.drop("ingestion_date") \
                                      .withColumn("title_norm", normalize_title_udf(col("name"))) \
                                      .withColumn("author_norm", normalize_author_udf(col("author"))) \
                                      .withColumn("ingestion_date", current_timestamp())

In [0]:
# Celda 16 - Write a Silver
df_bestsellers_final.write.mode("overwrite").insertInto(f"{catalogo}.{esquema_sink}.bestsellers_transformed")

In [0]:
# Celda 16.5 
from pyspark.sql import Window as W

dedup_window = W.partitionBy("title_norm").orderBy(F.desc("bbeScore"))
df_catalog_dedup = df_catalog.withColumn("rn", F.row_number().over(dedup_window)) \
                              .filter(col("rn") == 1) \
                              .drop("rn")

In [0]:
# Celda 17 
join_condition = (df_bestsellers_final["title_norm"] == df_catalog_dedup["title_norm"]) & (
    (F.instr(df_catalog_dedup["author_norm"], df_bestsellers_final["author_norm"]) > 0) |
    (F.instr(df_bestsellers_final["author_norm"], df_catalog_dedup["author_norm"]) > 0)
)

df_cross = df_bestsellers_final.alias("b").join(
    df_catalog_dedup.alias("c"), join_condition, "left"
)

In [0]:
# Celda 18
df_catalog_vs_bestsellers = df_cross.select(
    col("b.name").alias("bestseller_name"),
    col("b.author").alias("bestseller_author"),
    col("b.genre").alias("bestseller_genre"),
    col("b.year").alias("year"),
    col("c.title").isNotNull().alias("in_catalog"),
    col("c.title").alias("catalog_title"),
    col("c.primary_genre").alias("catalog_primary_genre"),
    col("c.rating").alias("catalog_rating")
).withColumn("ingestion_date", current_timestamp())

In [0]:
# Celda 19 - Write a Silver
df_catalog_vs_bestsellers.write.mode("overwrite").insertInto(f"{catalogo}.{esquema_sink}.catalog_vs_bestsellers")

In [0]:
# Celda 20
df_country = spark.table(f"{catalogo}.{esquema_source}.country_reading")

In [0]:
# Celda 21
window_spec = Window.orderBy(F.desc("books_read_annually"))
df_country_final = df_country.drop("ingestion_date") \
                              .withColumn("reading_rank", F.rank().over(window_spec)) \
                              .withColumn("ingestion_date", current_timestamp())

In [0]:
# Celda 22 - Write a Silver
df_country_final.write.mode("overwrite").insertInto(f"{catalogo}.{esquema_sink}.country_reading_transformed")